# Part 3 — Graph Initialization & Management

*Last updated:* 2026-01-08

This notebook shows how to **build and cache** an IDTrack graph snapshot for:
- `homo_sapiens` (human)
- `mus_musculus` (mouse)
- `sus_scrofa` (pig)

A graph build is the most expensive step. The good news:
- you usually do it **once per organism + snapshot boundary + external YAML configuration**
- the snapshot can be **multi-assembly** (human has overlapping GRCh38/GRCh37; mouse/pig are clean-handoff by release but legacy builds are supported)
- then you reuse the cached graph for fast conversions

**Learning objectives**
- Build (or load) a graph snapshot for each organism.
- Verify that the snapshot exists on disk.
- Learn practical graph-management habits (reload vs rebuild, cache hygiene).

> **Prerequisite:** run `02_prepare_new_external_yaml.ipynb` first (especially important for mouse and pig).


## 3.0 — What you should expect (time / disk)

Graph building can take:
- minutes to hours (depends on organism, enabled externals, and cache status)
- multiple GB of disk for cached tables + the graph pickle

Plan for this like you would plan for downloading a reference genome + annotation.


In [1]:
# Load notebook utilities (collapsible output magic for tutorials)
%load_ext _notebook_utils

In [2]:
# 1) Setup
from __future__ import annotations

import os
from pathlib import Path

import idtrack

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', './idtrack_cache')).resolve()
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api.configure_logger()

print('Local repository:', LOCAL_REPOSITORY)


Local repository: /Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache


## 3.0.1 — Sanity check: do your external YAML files exist?

Human has a packaged default, but for mouse and pig you should have local `*_externals_modified.yml` files.


In [3]:
# External YAML presence (created in Part 2)
human_yaml = LOCAL_REPOSITORY / 'homo_sapiens_externals_modified.yml'
mouse_yaml = LOCAL_REPOSITORY / 'mus_musculus_externals_modified.yml'
pig_yaml = LOCAL_REPOSITORY / 'sus_scrofa_externals_modified.yml'

HAS_HUMAN_YAML = human_yaml.exists()
HAS_MOUSE_YAML = mouse_yaml.exists()
HAS_PIG_YAML = pig_yaml.exists()

print(("OK" if HAS_HUMAN_YAML else "NOTE: missing (human can fall back to packaged default)").ljust(55), human_yaml.name)
print(("OK" if HAS_MOUSE_YAML else "MISSING (create in Part 2 for mouse)").ljust(55), mouse_yaml.name)
print(("OK" if HAS_PIG_YAML else "MISSING (create in Part 2 for pig)").ljust(55), pig_yaml.name)


OK                                                      homo_sapiens_externals_modified.yml
OK                                                      mus_musculus_externals_modified.yml
OK                                                      sus_scrofa_externals_modified.yml


If a file is missing:
- go back to `02_prepare_new_external_yaml.ipynb`
- generate the template and create the `_modified.yml` file


## 3.1–3.3 — Build graph snapshots (one per organism)

The canonical pattern is:

1. resolve organism name
2. pick snapshot release
3. (optional) choose a primary genome assembly for output (defaults to the newest/highest-priority assembly for that organism)
4. `api.build_graph(...)`
5. inspect + reuse

We do this for each organism below. If you only need one organism, run only that section.


### 3.1 — Human graph initialization (multi-assembly)

By default, the human snapshot is built with GRCh38 as the primary assembly (assembly code `38`), while also including GRCh37 (`37`) and
older archives when they exist within the snapshot window.

This is what enables atlas-building workflows where different datasets were annotated with different genome builds, but you want one unified
identifier space.


In [4]:
organism, latest_release = api.resolve_organism('human')
SNAPSHOT_RELEASE = latest_release  # pin to a specific release if needed
organism, SNAPSHOT_RELEASE


2026-01-11 21:05:28 INFO:verify_organism: Ensembl Rest API query to get the organism names and associated releases.


('homo_sapiens', 115)

In [5]:
%%collapse Click to show build logs
# Included for tutorial purposes only.

# Build (or load) the graph snapshot
# - calculate_caches=True speeds up later queries (slower build, faster use).
api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE, calculate_caches=False)


In [6]:
# Quick inspection
g = api.track.graph
print('Organism:', g.graph.get('organism'))
print('Snapshot release:', g.graph.get('ensembl_release'))
print('Main assembly:', g.graph.get('genome_assembly'))
print('Assemblies in this graph:', sorted(api.list_genome_assemblies()))
print('Nodes:', g.number_of_nodes())
print('Edges:', g.number_of_edges())

aed = sorted(getattr(g, 'available_external_databases', []))
print('External DBs enabled (count):', len(aed))
print('External DBs (first 20):', aed[:20])


Organism: homo_sapiens
Snapshot release: 115
Main assembly: 38
Assemblies in this graph: [36, 37, 38]
Nodes: 3682041


2026-01-11 22:08:48 INFO:the_graph: Cached properties being calculated: available_external_databases


Edges: 8615426
External DBs enabled (count): 27
External DBs (first 20): ['CCDS', 'Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'HGNC Symbol', 'Havana gene', 'Havana transcript', 'Havana translation', 'NCBI gene', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RefSeq_mRNA', 'RefSeq_mRNA_predicted', 'RefSeq_ncRNA', 'RefSeq_ncRNA_predicted', 'RefSeq_peptide', 'RefSeq_peptide_predicted', 'UniProtKB Gene Name', 'Uniprot/SPTREMBL', 'Uniprot/SWISSPROT']


In [7]:
# Where is the graph file stored?
sorted(LOCAL_REPOSITORY.glob('graph_homo_sapiens*.pickle'))[-5:]


[PosixPath('/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/graph_homo_sapiens_min48_max115_narrow.pickle')]

### 3.2 — Mouse graph initialization (clean handoff)

Mouse is a clean-handoff species (one maintained assembly per release: GRCm37 → GRCm38 → GRCm39). Older assemblies mainly matter for
legacy datasets and archive releases; you typically do not have overlapping assemblies within the same release.


In [4]:
organism, latest_release = api.resolve_organism('mus musculus')
SNAPSHOT_RELEASE = latest_release
organism, SNAPSHOT_RELEASE


2026-01-11 22:44:22 INFO:verify_organism: Ensembl Rest API query to get the organism names and associated releases.


('mus_musculus', 115)

In [5]:
%%collapse Click to show build logs
# Included for tutorial purposes only.

# Build (or load) the mouse graph snapshot
if HAS_MOUSE_YAML:
    api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE, calculate_caches=False)
else:
    print('Skipping mouse build: mus_musculus_externals_modified.yml is missing (run Part 2 first).')


In [6]:
if HAS_MOUSE_YAML:
    g = api.track.graph
    print('Organism:', g.graph.get('organism'))
    print('Snapshot release:', g.graph.get('ensembl_release'))
    print('Main assembly:', g.graph.get('genome_assembly'))
    print('Assemblies in this graph:', sorted(api.list_genome_assemblies()))
    print('Nodes:', g.number_of_nodes())
    print('Edges:', g.number_of_edges())

    aed = sorted(getattr(g, 'available_external_databases', []))
    print('External DBs enabled (count):', len(aed))
    print('External DBs (first 20):', aed[:20])
else:
    print('Mouse graph not built (missing YAML).')


Organism: mus_musculus
Snapshot release: 115
Main assembly: 39
Assemblies in this graph: [37, 38, 39]
Nodes: 2476278


2026-01-11 23:58:19 INFO:the_graph: Cached properties being calculated: available_external_databases


Edges: 5367328
External DBs enabled (count): 28
External DBs (first 20): ['CCDS', 'Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'Havana gene', 'Havana transcript', 'Havana translation', 'MGI Symbol', 'NCBI gene', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RefSeq_mRNA', 'RefSeq_mRNA_predicted', 'RefSeq_ncRNA', 'RefSeq_ncRNA_predicted', 'RefSeq_peptide', 'RefSeq_peptide_predicted', 'UniProtKB Gene Name', 'Uniprot/SPTREMBL', 'Uniprot/SWISSPROT']


In [7]:
sorted(LOCAL_REPOSITORY.glob('graph_mus_musculus*.pickle'))[-5:]


[PosixPath('/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/graph_mus_musculus_min48_max115_narrow.pickle')]

### 3.3 — Pig graph initialization (clean handoff)

Pig is a clean-handoff species (one maintained assembly per release: Sscrofa9.2 → Sscrofa10.2 → Sscrofa11.1). Older assemblies mainly
matter for legacy datasets and archive releases; you typically do not have overlapping assemblies within the same release.


In [8]:
organism, latest_release = api.resolve_organism('sus scrofa')
SNAPSHOT_RELEASE = latest_release
organism, SNAPSHOT_RELEASE


2026-01-11 23:58:21 INFO:verify_organism: Ensembl Rest API query to get the organism names and associated releases.


('sus_scrofa', 115)

In [9]:
%%collapse Click to show build logs
# Included for tutorial purposes only.

# Build (or load) the pig graph snapshot
if HAS_PIG_YAML:
    api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE, calculate_caches=False)
else:
    print('Skipping pig build: sus_scrofa_externals_modified.yml is missing (run Part 2 first).')


In [10]:
if HAS_PIG_YAML:
    g = api.track.graph
    print('Organism:', g.graph.get('organism'))
    print('Snapshot release:', g.graph.get('ensembl_release'))
    print('Main assembly:', g.graph.get('genome_assembly'))
    print('Assemblies in this graph:', sorted(api.list_genome_assemblies()))
    print('Nodes:', g.number_of_nodes())
    print('Edges:', g.number_of_edges())

    aed = sorted(getattr(g, 'available_external_databases', []))
    print('External DBs enabled (count):', len(aed))
    print('External DBs (first 20):', aed[:20])
else:
    print('Pig graph not built (missing YAML).')


Organism: sus_scrofa
Snapshot release: 115
Main assembly: 111
Assemblies in this graph: [9, 102, 111]
Nodes: 1073214


2026-01-12 00:24:32 INFO:the_graph: Cached properties being calculated: available_external_databases


Edges: 1757774
External DBs enabled (count): 24
External DBs (first 20): ['Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'HGNC Symbol', 'Havana gene', 'Havana transcript', 'NCBI gene', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RefSeq_mRNA', 'RefSeq_mRNA_predicted', 'RefSeq_ncRNA', 'RefSeq_ncRNA_predicted', 'RefSeq_peptide', 'RefSeq_peptide_predicted', 'UniProtKB Gene Name', 'Uniprot/SPTREMBL', 'Uniprot/SWISSPROT', 'VGNC Symbol', 'synonym_id::EntrezGene']


In [11]:
sorted(LOCAL_REPOSITORY.glob('graph_sus_scrofa*.pickle'))[-5:]


[PosixPath('/Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache/graph_sus_scrofa_min48_max115_narrow.pickle')]

## 3.4 — Graph management (all species)

### Reloading vs rebuilding

- `api.build_graph(...)` is safe to call repeatedly.
  - If the snapshot already exists on disk, IDTrack will **load** it.
  - If it does not exist yet, IDTrack will **build** it (slow, first-time only).

### Cache hygiene

Your local repository can accumulate:
- downloaded tables
- graph snapshot pickle files
- intermediate files used during builds

> **Tip:** Treat your local repository as *project infrastructure*. Keep it stable so you get the benefits of caching.

### Switching organisms / assemblies

A graph snapshot is specific to:
- organism
- snapshot boundary (max release)
- external YAML contents
- the chosen **primary assembly** (the default output coordinate system)

Even though the snapshot can include multiple assemblies, changing the primary assembly changes the snapshot and requires a rebuild.

> **Tip:** The cached graph filename does not include the assembly. If you want to keep two different primary assemblies side-by-side, use
> separate local repositories (or copy the graph pickle file).

### Performance tips

- Use `calculate_caches=True` during builds when you plan to do many conversions afterward.
- Keep your external YAML allowlist small to reduce ambiguity and search space.

> **Warning:** Do not delete caches unless you understand the consequence (you may force a full rebuild).


In [12]:
# Helper: list what IDTrack has cached in your local repository.
# Safe: this does NOT delete anything.

from pathlib import Path

cache = LOCAL_REPOSITORY

print('Local repository:', cache)

patterns = [
    'graph_*.pickle',
    '*_externals_modified.yml',
    '*_externals_template.yml',
]

for pat in patterns:
    hits = sorted(cache.glob(pat))
    print()
    print(f'{pat} ({len(hits)}):')
    for p in hits[:10]:
        print('  ', p.name)
    if len(hits) > 10:
        print('  ...')


Local repository: /Users/kemalinecik/git_nosync/master_idtrack/idtrack/docs/_notebooks/idtrack_cache

graph_*.pickle (3):
   graph_homo_sapiens_min48_max115_narrow.pickle
   graph_mus_musculus_min48_max115_narrow.pickle
   graph_sus_scrofa_min48_max115_narrow.pickle

*_externals_modified.yml (3):
   homo_sapiens_externals_modified.yml
   mus_musculus_externals_modified.yml
   sus_scrofa_externals_modified.yml

*_externals_template.yml (3):
   homo_sapiens_externals_template.yml
   mus_musculus_externals_template.yml
   sus_scrofa_externals_template.yml
